# IMDb sentiment analysis: TextCNN and TF-IDF baselines

**Ibrakhim Razyyev · Academic portfolio project**

Origin: practical work for AI Fundamentals based on Huawei course materials.
This version organises the TextCNN experiment into a reproducible project and
compares it with Naive Bayes and a linear SVM on the **same IMDb split**.

The original Colab notebook also contained an RT-polarity experiment. It is not
mixed into this benchmark because scores from different datasets are not directly comparable.


## 1. Setup

Run all cells in order. In Colab, this cell downloads the project and installs its
dependencies. Locally, open this notebook from the cloned repository after installing
`requirements-dev.txt`. No GPU session is required to explore the committed results.


In [ ]:
from pathlib import Path
import os
import subprocess
import sys

if Path('/content').is_dir():
    root = Path('/content/imdb-sentiment-analysis')
    if not (root / 'sentiment.py').exists():
        subprocess.check_call(['git', 'clone', '--depth', '1',
            'https://github.com/ibra-0/imdb-sentiment-analysis.git', str(root)])
    os.chdir(root)
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements.txt'])
elif (Path.cwd().parent / 'sentiment.py').exists():
    os.chdir(Path.cwd().parent)

assert Path('sentiment.py').exists(), 'Open the notebook from the project repository.'
print('Project ready')


## 2. Data and evaluation protocol

IMDb has 25,000 labelled training reviews and 25,000 labelled test reviews.
We split the training portion into **20,000 training / 5,000 validation** reviews
with stratification and seed 42, and retain the official test split.

Only training texts are used to build TF-IDF features and the CNN vocabulary.
The CNN checkpoint is selected by validation loss. Accuracy, macro F1 and a
confusion matrix are reported on the test set. Dataset revision and split hashes
are recorded in `results/metrics.json`.


In [ ]:
import json
from dataclasses import asdict
from sentiment import Config, DATASET, DATASET_REVISION, clean_text, TextCNN

config = Config()
print('Dataset:', DATASET)
print('Revision:', DATASET_REVISION)
print('Configuration:', asdict(config))
print('Preprocessing:', clean_text('<br />A GREAT movie!'))


## 3. TextCNN architecture

Tokens → 128-dimensional learned embeddings → parallel 1D convolutions with
widths 3, 4 and 5 (100 filters each) → ReLU → max pooling → dropout → two logits.
Reviews are truncated/padded to 256 tokens; the vocabulary is capped at 20,000.
The original architecture is retained, including max pooling over padded positions.
Padding-aware pooling and longer context are possible follow-up experiments.


In [ ]:
import inspect
print(inspect.getsource(TextCNN))


## 4. Reproduce training (optional)

Set `RUN_TRAINING = True` to download IMDb and train all three models.
This may take several minutes or longer depending on hardware. The default
only explores committed results and does not claim to rerun training.
New checkpoints and reports are saved under the git-ignored `runs/` directory.


In [ ]:
RUN_TRAINING = False

if RUN_TRAINING:
    from sentiment import train
    report = train(config, Path('runs/notebook'))
    report_dir = Path('runs/notebook')
else:
    report_dir = Path('results')
    report = json.loads((report_dir / 'metrics.json').read_text())

assert not report['smoke_run'], 'Use a full experiment for reported portfolio results.'
print('Mode:', 'new training run' if RUN_TRAINING else 'committed experiment results')
print('Device used for the recorded run:', report['device'])
print('Split counts:', {k: v['count'] for k, v in report['split_manifest'].items()})


## 5. Results

All rows below use the same training/validation/test examples. The TF-IDF baselines
use uni/bigrams and up to 20,000 features. The CNN learns its own embeddings.
This is a small educational benchmark, not an exhaustive hyperparameter search.


In [ ]:
print(f"{'Model':30} {'Val accuracy':>14} {'Test accuracy':>15} {'Test macro F1':>15}")
for name, result in report['results'].items():
    print(f"{name:30} {result['validation']['accuracy']:14.4f} "
          f"{result['test']['accuracy']:15.4f} {result['test']['macro_f1']:15.4f}")


In [ ]:
from IPython.display import SVG, display
display(SVG(filename=str(report_dir / 'learning_curves.svg')))
display(SVG(filename=str(report_dir / 'confusion_matrix.svg')))


## 6. Example predictions and limitations

These are the predictions saved during the recorded training run. They are
illustrations, not an independent test set. Softmax scores are not calibrated
probabilities of correctness. Short reviews, negation, sarcasm, unfamiliar words
and text after token 256 can cause errors. This model is trained on English movie
reviews and is not validated for Russian, Kazakh or general customer feedback.


In [ ]:
for example in report['examples']:
    print(example['text'])
    print(f"  {example['label']}; model score={example['score']:.3f}; "
          f"truncated={example['truncated']}")


## 7. Predict a new review after training

Run the following cell after enabling training, or train from the terminal first:

```bash
python sentiment.py train --output runs/full
python sentiment.py predict --checkpoint runs/full/textcnn.pt --text "A thoughtful and enjoyable movie."
```

Weights are generated locally rather than committed. The command-line path
reuses exactly the same cleaning, vocabulary and model configuration as training.


In [ ]:
checkpoint_path = Path('runs/notebook/textcnn.pt')
if checkpoint_path.exists():
    subprocess.check_call([sys.executable, 'sentiment.py', 'predict',
        '--checkpoint', str(checkpoint_path), '--text', 'A thoughtful and enjoyable movie.'])
else:
    print('Train first to run inference on a new review.')


## References and provenance

- [IMDb dataset](https://ai.stanford.edu/~amaas/data/sentiment/), Maas et al. (2011).
- [Hugging Face dataset mirror](https://huggingface.co/datasets/stanfordnlp/imdb).
- [Original course notebook](https://colab.research.google.com/drive/1CMyEXJOfwXgATq-iDl6A1154DYiPLHeZ)
  (access depends on its sharing settings).
- Kim (2014), [Convolutional Neural Networks for Sentence Classification](https://aclanthology.org/D14-1181/),
  for the TextCNN architecture family. This is a compact educational implementation,
  not a reproduction of that paper's reported benchmark.

Portfolio preparation added modular code, a shared-split benchmark, reproducibility
metadata, a CLI, documentation and checks with AI assistance. Training results and
limitations are recorded explicitly. No claim of commercial deployment is made.
